# Hyperspectral Image Fusion: Chikusei Dataset

Training NullFusion v4 and KrylovNet on Chikusei (128 bands).

## Dataset Info
- **Sensor**: Headwall Nano-Hyperspec-VNIR-C
- **Bands**: 128 (363-1018 nm)
- **Spatial**: 2517 x 2335 pixels
- **Source**: [Kaggle - mingliu123/chikusei](https://www.kaggle.com/datasets/mingliu123/chikusei)

## SOTA Targets (Chikusei x4)
| Method | PSNR | SAM |
|--------|------|-----|
| CoFusion (2026) | 49.14 | 2.60 |
| RAMoE (2026) | 48.10 | 0.79 |
| SMGU-Net (2025) | 48.82 | 2.72 |
| PSRT (2023) | 47.99 | 2.84 |
| KrylovNet v1 (ours) | 43.69 | 6.07 |

In [ ]:
# Install dependencies
!pip install -q einops h5py

In [ ]:
import os
import sys
import json
import time
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.io import loadmat
from scipy.ndimage import convolve, uniform_filter
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Dataset paths (Kaggle)
CHIKUSEI_ROOT = "/kaggle/input/chikusei"
OUTPUT_DIR = "/kaggle/working"

# Check dataset
import glob
mat_files = glob.glob(os.path.join(CHIKUSEI_ROOT, "**", "*.mat"), recursive=True)
print(f"Found {len(mat_files)} .mat files")
for f in mat_files[:5]:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")

## 1. SRF (Spectral Response Function)

In [ ]:
def chikusei_srf(bands=128, normalise=True):
    """Chikusei Headwall Nano-Hyperspec response as [bands, 3]."""
    wl = np.linspace(363.0, 1018.0, 128)
    raw = np.stack([
        np.exp(-((wl - 620.0) ** 2) / (2 * 80.0 ** 2)),  # Red
        np.exp(-((wl - 540.0) ** 2) / (2 * 70.0 ** 2)),  # Green
        np.exp(-((wl - 460.0) ** 2) / (2 * 60.0 ** 2)),  # Blue
    ], axis=1).astype(np.float32)
    if bands != 128:
        xs = np.linspace(0.0, 1.0, 128)
        xd = np.linspace(0.0, 1.0, bands)
        raw = np.stack([np.interp(xd, xs, raw[:, i]) for i in range(3)], axis=1)
    if normalise:
        raw = raw / np.maximum(raw.sum(axis=0, keepdims=True), 1e-8)
    return raw.astype(np.float32)

def conditioning(srf):
    """SRF conditioning analysis."""
    s = np.linalg.svd(srf, compute_uv=False)
    return {"cond": float(s[0] / max(s[-1], 1e-12))}

srf = chikusei_srf(128)
info = conditioning(srf)
print(f"SRF shape: {srf.shape}")
print(f"Condition number: {info['cond']:.2f}")

# Plot SRF
wl = np.linspace(363, 1018, 128)
plt.figure(figsize=(10, 4))
plt.plot(wl, srf[:, 0], 'r-', label='Red')
plt.plot(wl, srf[:, 1], 'g-', label='Green')
plt.plot(wl, srf[:, 2], 'b-', label='Blue')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Response')
plt.title('Chikusei Nano-Hyperspec SRF (128 bands)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Dataset Loader

In [ ]:
def gaussian_kernel2d(size=9, sigma=1.2):
    ax = np.arange(size, dtype=np.float32) - (size - 1) / 2.0
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-0.5 * (xx**2 + yy**2) / (sigma**2))
    return (k / k.sum()).astype(np.float32)

class ChikuseiDataset(Dataset):
    def __init__(self, root, split='train', bands=128, scale=4, patch_size=64):
        self.split = split
        self.bands = bands
        self.scale = scale
        self.patch_size = patch_size
        self.srf = chikusei_srf(bands)
        self.kernel = gaussian_kernel2d(9, 1.2)
        
        # Load cube
        mat_files = glob.glob(os.path.join(root, '**', '*.mat'), recursive=True)
        mat_files.sort(key=lambda f: os.path.getsize(f), reverse=True)
        
        print(f"Loading from: {mat_files[0]}")
        data = loadmat(mat_files[0])
        for key, val in data.items():
            if not key.startswith('__') and hasattr(val, 'shape'):
                arr = np.array(val, dtype=np.float32)
                if arr.ndim == 3 and min(arr.shape) > 10:
                    if arr.shape[0] > arr.shape[-1]:
                        arr = arr.transpose(2, 0, 1)
                    if arr.max() > 1.0:
                        arr = arr / arr.max()
                    self.cube = arr
                    break
        
        C, H, W = self.cube.shape
        print(f"Cube: {C} bands, {H}x{W} pixels")
        
        # Split into patches
        p = patch_size
        coords = [(y, x) for y in range(0, H - p + 1, p) 
                         for x in range(0, W - p + 1, p)]
        random.seed(42)
        random.shuffle(coords)
        n_train = int(0.7 * len(coords))
        self.patches = coords[:n_train] if split == 'train' else coords[n_train:]
        print(f"{split}: {len(self.patches)} patches")
    
    def __len__(self):
        return len(self.patches) * (50 if self.split == 'train' else 1)
    
    def _sim(self, gt):
        C, H, W = gt.shape
        blurred = np.empty_like(gt)
        for c in range(C):
            blurred[c] = convolve(gt[c], self.kernel, mode='wrap')
        hr, wr = H // self.scale, W // self.scale
        y0 = (H - hr * self.scale) // 2
        x0 = (W - wr * self.scale) // 2
        lr = blurred[:, y0::self.scale, x0::self.scale].astype(np.float32)
        msi = np.einsum('chw,cm->mhw', gt, self.srf).astype(np.float32)
        return lr, np.clip(msi, 0, 1)
    
    def __getitem__(self, idx):
        y, x = self.patches[idx % len(self.patches)]
        p = self.patch_size
        gt = self.cube[:, y:y+p, x:x+p].copy()
        if self.split == 'train':
            if random.random() < 0.5:
                gt = gt[:, :, ::-1].copy()
            if random.random() < 0.5:
                gt = gt[:, ::-1, :].copy()
        lr, msi = self._sim(gt)
        return (torch.from_numpy(gt), torch.from_numpy(lr), torch.from_numpy(msi))

In [ ]:
# Test dataset
train_ds = ChikuseiDataset(CHIKUSEI_ROOT, 'train', 128, 4, 64)
test_ds = ChikuseiDataset(CHIKUSEI_ROOT, 'test', 128, 4, 64)

# Visualize sample
gt, lr, msi = train_ds[0]
print(f"GT: {gt.shape}, LR: {lr.shape}, MSI: {msi.shape}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gt[30].numpy(), cmap='viridis')
axes[0].set_title(f'GT (band 30)')
axes[1].imshow(lr[30].numpy(), cmap='viridis')
axes[1].set_title(f'LR-HSI (band 30)')
axes[2].imshow(msi.numpy().transpose(1, 2, 0))
axes[2].set_title('MSI (RGB)')
plt.tight_layout()
plt.show()

## 3. Metrics

In [ ]:
def calc_psnr(pred, gt):
    mse = np.mean((pred - gt) ** 2)
    return 100.0 if mse < 1e-12 else -10.0 * np.log10(mse)

def calc_sam(pred, gt):
    p = pred.reshape(pred.shape[0], -1)
    g = gt.reshape(gt.shape[0], -1)
    p = p / (np.linalg.norm(p, axis=0, keepdims=True) + 1e-8)
    g = g / (np.linalg.norm(g, axis=0, keepdims=True) + 1e-8)
    cos = np.clip((p * g).sum(0), -1.0, 1.0)
    return np.mean(np.arccos(cos)) * 180.0 / np.pi

def calc_ssim(pred, gt):
    C1, C2 = (0.01)**2, (0.03)**2
    mu1 = uniform_filter(pred, size=3, mode='reflect')
    mu2 = uniform_filter(gt, size=3, mode='reflect')
    sigma12 = uniform_filter(pred * gt, size=3, mode='reflect') - mu1 * mu2
    sigma1 = uniform_filter(pred**2, size=3, mode='reflect') - mu1**2
    sigma2 = uniform_filter(gt**2, size=3, mode='reflect') - mu2**2
    return np.mean(((2*mu1*mu2 + C1) * (2*sigma12 + C2)) / 
                   ((mu1**2 + mu2**2 + C1) * (sigma1 + sigma2 + C2) + 1e-8))

def calc_ergas(pred, gt, scale=4):
    C = pred.shape[0]
    err = (pred - gt) ** 2
    ergas = sum(err[c].mean() / (gt[c].mean()**2 + 1e-8) for c in range(C))
    return math.sqrt(ergas / C) * 100.0 * scale

## 4. NullFusion v4 Model (128 bands)

In [ ]:
# [Paste the NullFusionNetV4Chikusei class from train_nullfusion_chikusei.py here]
# Or import it:
# from experiments.scripts.train_nullfusion_chikusei import NullFusionNetV4Chikusei

# For now, we'll define a simplified version for testing
class SimplifiedNullFusion(nn.Module):
    def __init__(self, bands=128, msi_bands=3, width=80):
        super().__init__()
        self.bands = bands
        # Simple encoder-decoder with null-space projection
        self.encoder = nn.Sequential(
            nn.Conv2d(bands + msi_bands, width, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(width, width, 3, 1, 1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(width, width, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(width, bands, 3, 1, 1),
        )
    
    def forward(self, lr, msi):
        # Upsample LR-HSI
        x_up = F.interpolate(lr, scale_factor=4, mode='bicubic', align_corners=False)
        msi_up = F.interpolate(msi, size=x_up.shape[-2:], mode='nearest')
        inp = torch.cat([x_up, msi_up], dim=1)
        h = self.encoder(inp)
        out = self.decoder(h) + x_up
        return {'out': out.clamp(0, 1)}

## 5. Training Loop

In [ ]:
def train_model(model, train_ds, test_ds, epochs=1000, eval_every=50, 
                batch_size=4, lr=2e-4, device='cuda'):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    l1 = nn.L1Loss()
    scaler = torch.cuda.amp.GradScaler(enabled=True)
    
    best_psnr = 0
    history = {'train_loss': [], 'test_psnr': [], 'test_sam': []}
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0
        t0 = time.time()
        
        for _ in range(100):  # steps per epoch
            idxs = [random.randrange(len(train_ds)) for _ in range(batch_size)]
            gts = torch.stack([train_ds[i][0] for i in idxs]).to(device)
            lhs = torch.stack([train_ds[i][1] for i in idxs]).to(device)
            mss = torch.stack([train_ds[i][2] for i in idxs]).to(device)
            
            with torch.cuda.amp.autocast(enabled=True):
                out = model(lhs, mss)['out']
                loss = l1(out, gts)
            
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            epoch_loss += loss.item()
        
        scheduler.step()
        history['train_loss'].append(epoch_loss / 100)
        
        if epoch % 10 == 0:
            dt = time.time() - t0
            print(f"Epoch {epoch:4d} | Loss {history['train_loss'][-1]:.5f} | {dt:.1f}s")
        
        if epoch % eval_every == 0:
            model.eval()
            psnrs, sams = [], []
            with torch.no_grad():
                for i in range(min(len(test_ds), 20)):
                    g, l, m = test_ds[i]
                    pred = model(l.unsqueeze(0).to(device), m.unsqueeze(0).to(device))['out'][0].cpu().numpy()
                    gt_np = g.numpy()
                    psnrs.append(calc_psnr(pred, gt_np))
                    sams.append(calc_sam(pred, gt_np))
            
            m_psnr = np.mean(psnrs)
            m_sam = np.mean(sams)
            history['test_psnr'].append(m_psnr)
            history['test_sam'].append(m_sam)
            
            marker = " [BEST]" if m_psnr > best_psnr else ""
            if m_psnr > best_psnr:
                best_psnr = m_psnr
                torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"  Test@{epoch}: PSNR {m_psnr:.4f} | SAM {m_sam:.3f}{marker}")
    
    return history

## 6. Run Training

In [ ]:
# Initialize model
model = SimplifiedNullFusion(bands=128, msi_bands=3, width=80)
nparams = sum(p.numel() for p in model.parameters())
print(f"Model params: {nparams/1e6:.2f}M")

# Train
device = 'cuda' if torch.cuda.is_available() else 'cpu'
history = train_model(model, train_ds, test_ds, epochs=500, eval_every=50,
                      batch_size=4, lr=2e-4, device=device)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('L1 Loss')

axes[1].plot(history['test_psnr'])
axes[1].set_title('Test PSNR')
axes[1].set_xlabel('Eval Step')
axes[1].set_ylabel('PSNR (dB)')

axes[2].plot(history['test_sam'])
axes[2].set_title('Test SAM')
axes[2].set_xlabel('Eval Step')
axes[2].set_ylabel('SAM (deg)')

plt.tight_layout()
plt.show()

# Final results
print(f"\nFinal PSNR: {history['test_psnr'][-1]:.4f}")
print(f"Final SAM: {history['test_sam'][-1]:.3f}")

## 7. SOTA Comparison

In [ ]:
sota = {
    'CoFusion (2026)': {'psnr': 49.14, 'sam': 2.60},
    'RAMoE (2026)': {'psnr': 48.10, 'sam': 0.79},
    'SMGU-Net (2025)': {'psnr': 48.82, 'sam': 2.72},
    'PSRT (2023)': {'psnr': 47.99, 'sam': 2.84},
    'U2Net (2023)': {'psnr': 47.93, 'sam': 2.77},
    'KrylovNet v1 (ours)': {'psnr': 43.69, 'sam': 6.07},
}

our_psnr = history['test_psnr'][-1]
our_sam = history['test_sam'][-1]

print("=" * 60)
print("Chikusei x4 SOTA Comparison")
print("=" * 60)
for name, vals in sota.items():
    delta_psnr = our_psnr - vals['psnr']
    delta_sam = our_sam - vals['sam']
    print(f"{name:25s} PSNR {vals['psnr']:6.2f} (Δ={delta_psnr:+.2f}) SAM {vals['sam']:5.2f} (Δ={delta_sam:+.2f})")
print("-" * 60)
print(f"{'Our Model':25s} PSNR {our_psnr:6.2f}         SAM {our_sam:5.2f}")

In [ ]:
# Save results
results = {
    'dataset': 'Chikusei',
    'bands': 128,
    'protocol': 'Chikusei x4, Sensor SRF, Wald blur',
    'params_M': nparams / 1e6,
    'final_psnr': float(history['test_psnr'][-1]),
    'final_sam': float(history['test_sam'][-1]),
    'sota_comparison': sota,
}

with open(os.path.join(OUTPUT_DIR, 'chikusei_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to chikusei_results.json")